Instalação de pacotes, seguido de importação de bibliotecas e funções necessárias.

In [7]:
# Instalar ou atualizar bibliotecas necessárias: datasets do Hugging Face, transformers, torch e google.
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub

# Importar biblioteca pandas.
import pandas as pd
import torch
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 129.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 139.2 MB/s eta 0:00:00


Processar o arquivo de base com as perguntas.

In [2]:
# Carregar as questões a serem usadas, cuja lógica está em outro arquivo.
# Realizar download do arquivo direto do GitHub.
!wget https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb -O questions.ipynb

# Executar o notebook de base na íntegra.
%run questions.ipynb

--2026-04-04 09:17:02--  https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9811 (9.6K) [text/plain]
Saving to: ‘questions.ipynb’

questions.ipynb     100%[===================>]   9.58K  --.-KB/s    in 0s      

2026-04-04 09:17:02 (95.7 MB/s) - ‘questions.ipynb’ saved [9811/9811]



Fazer logon no hugging face.

In [10]:
# Resgata o segredo cadastrado com o nome 'hugging_colab'
def logon_hugging_face():
  try:
    # Recuperar valor da chave criada no huggin face e cadastrada no Secrets do Google Colab.
    hf_token = userdata.get('hugging_colab')
    # Realiza o login no Hugging Face
    login(token=hf_token)
  except Exception as e:
    print(f"Erro ao recuperar token: {e}")

def markdown_prepare(papel, categoria, contexto, dificuldade, instrucao):
  # Preparação do prompt em Markdown
  prompt_usuario = f"""
  ### Papel:
  {papel}

  ### Área de atuação:
  {categoria}

  ### Contexto:
  {contexto}

  ### Nível de Dificuldade:
  Com valores de 1 a 4, onde 1 indica o mais simples e 4 o mais complexo.
  {dificuldade}

  ### Instrução:
  {instrucao}
  """
  # Retornar o prompt em markdown.
  return prompt_usuario

Modelo baseado no llama 3, especializado em direito brasileiro e com fine tunning.

In [12]:
logon_hugging_face()

# Modelo do hugging face no domínio específico.
model_id = "Jurema-br/Jurema-7B"

# Configuração de BitsAndBytes.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

# Token
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Criar uma lista vazia, para guardar as respostas, por questão de performance.
responses = []

# Repetição para percorrer todo Dataframe.
for index, row in df_my_questions.iterrows():
  # preenchimento dos parâmetros da pergunta, com base na linha corrente.
  questao = row['question']
  papel = row['system']
  categoria = row['category']
  contexto = row['statement']
  dificuldade = row['level']
  instrucao = row['turns']

  # Chat template.
  messages = [
    {
      "role": "system",
      "content": {papel}
    },
    {"role": "user", "content": prompt_usuario}
  ]

  # Gera o prompt formatado com os tokens especiais do modelo.
  prompt_formatado = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
  )

  inputs = tokenizer(prompt_formatado, return_tensors="pt").to("cuda")

  # Geração da Resposta
  # Aumentei max_new_tokens pois 50 costuma ser muito pouco para respostas complexas
  with torch.no_grad():
    outputs = model.generate(
      **inputs,
      max_new_tokens=256,
      eos_token_id=tokenizer.eos_token_id,
      pad_token_id=tokenizer.pad_token_id,
      do_sample=True,
      temperature=0.1
    )

    # Decodificação
    # Cortamos o input da resposta para pegar apenas o que o modelo gerou
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Como usamos chat_template, a resposta costuma vir após o marcador de assistente
    # Uma forma simples de limpar no Llama 3:
    clear_response = full_response.split("assistant")[-1].strip()

    responses.append({"question": row['question'], "response": clear_response})
    print(f"Questão {index} processada.")

# Resultado final
df_response = pd.DataFrame(responses)

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/Jurema-br/Jurema-7B.
403 Client Error. (Request ID: Root=1-69d0df09-2efabf864fb6466d3d58c2a7;7a60a144-f4df-4760-90f7-ddcf9381cbf0)

Cannot access gated repo for url https://huggingface.co/Jurema-br/Jurema-7B/resolve/main/config.json.
Access to model Jurema-br/Jurema-7B is restricted and you are not in the authorized list. Visit https://huggingface.co/Jurema-br/Jurema-7B to ask for access.